# Chapter 6, Guide 1: Grover's algorithm

Companion notebook to *Quantum Algorithms from First Principles* (Muhammad Faryad),
Chapter 6, "Implementation guide: Building Grover's algorithm", and the testing box
"Testing Grover, and what to expect from hardware".

The notebook follows the guide in order: qubits, the oracle for a known target $w$,
the diffuser in the form that is actually coded, the choice of $r$, and then the four
tests in the order the book gives them.

**Convention.** The book writes kets as $|q_0 q_1 \cdots q_{n-1}\rangle$ with $q_0$ the
most significant bit and the top wire. Qiskit numbers qubits the other way (qubit 0 is
the least significant bit and prints on the right). Every notebook in this repository
therefore maps book wire $j$ to Qiskit qubit $n-1-j$ through the helper `bq(j)`, so that
Qiskit's bitstrings and statevector indices read exactly as the book's.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator

np.set_printoptions(precision=4, suppress=True)


def bq(j, n):
    """Book wire j (0 = top, most significant) -> Qiskit qubit index."""
    return n - 1 - j

## Qubits

$n$ search qubits plus one ancilla prepared in $|-\rangle$ for the oracle. Qiskit's
multi-controlled $X$ (`mcx`) synthesizes its own Toffoli chain, so the $n-2$ clean
ancillas the book counts are hidden inside the transpiler; the gate counts at the end
of the notebook expose them.

## The oracle for a known target

A multi-controlled $X$ onto the $|-\rangle$ ancilla, with control polarities read off
the bits of $w$. Phase kickback turns the bit flip into a sign.

In [2]:
def oracle_known_target(n, w):
    """U_f for f(x) = [x == w] on n search qubits + 1 ancilla (book wire n)."""
    qc = QuantumCircuit(n + 1, name=f"U_f(w={w})")
    bits = [(w >> (n - 1 - j)) & 1 for j in range(n)]   # book order: bits[0] is the MSB
    # set polarities: control on |0> needs an X before and after
    for j, b in enumerate(bits):
        if b == 0:
            qc.x(bq(j, n + 1))
    qc.mcx([bq(j, n + 1) for j in range(n)], bq(n, n + 1))
    for j, b in enumerate(bits):
        if b == 0:
            qc.x(bq(j, n + 1))
    return qc


def diffuser(n):
    """V = -H^n X^n C^{n-1}(Z) X^n H^n, coded as the book's Fig. 'diffuser as actually coded'.
    This circuit realizes -V; the sign is a global phase for plain Grover."""
    qc = QuantumCircuit(n, name="diffuser")
    qc.h(range(n))
    qc.x(range(n))
    # C^{n-1}(Z): multi-controlled X between two Hadamards on the target (book wire n-1)
    t = bq(n - 1, n)
    qc.h(t)
    qc.mcx([bq(j, n) for j in range(n - 1)], t)
    qc.h(t)
    qc.x(range(n))
    qc.h(range(n))
    return qc


print(diffuser(3).draw())

     ┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐┌───┐
q_0: ┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     ├───┤├───┤└───┘└─┬─┘├───┤├───┤└───┘
q_1: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤├───┤       │  ├───┤├───┤     
q_2: ┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     └───┘└───┘          └───┘└───┘     


## Test 1: $U_f$ is diagonal with $-1$ exactly at the solution

Prepare the ancilla in $|-\rangle$, apply the oracle, and look at the induced action on
the search register. Equivalently, build the $2^n \times 2^n$ matrix of the phase
oracle $U_f = I - 2|w\rangle\langle w|$ and compare.

In [3]:
def phase_oracle_matrix(n, w):
    """Extract the 2^n x 2^n phase oracle acting on the search register from the
    ancilla-based bit-flip oracle, by conjugating with the |-> ancilla."""
    qc = QuantumCircuit(n + 1)
    a = bq(n, n + 1)
    qc.x(a); qc.h(a)                 # ancilla -> |->
    qc.compose(oracle_known_target(n, w), inplace=True)
    qc.h(a); qc.x(a)                 # ancilla back to |0>
    U = Operator(qc).data            # (2^{n+1} x 2^{n+1}), ancilla is Qiskit qubit 0
    # keep the block with ancilla = 0: rows/cols with even index (Qiskit LSB = ancilla)
    return U[0::2, 0::2]


n, w = 3, 5
Uf = phase_oracle_matrix(n, w)
expected = np.eye(2**n); expected[w, w] = -1
print("diagonal:", np.allclose(Uf, np.diag(np.diag(Uf))))
print("diag entries:", np.real(np.diag(Uf)))
assert np.allclose(Uf, expected), "oracle wrong"
print("Test 1 passed: U_f = I - 2|w><w| with w =", w)

diagonal: True
diag entries: [ 1.  1.  1.  1.  1. -1.  1.  1.]
Test 1 passed: U_f = I - 2|w><w| with w = 5


## Test 2: $V = 2|s\rangle\langle s| - I$ numerically

The coded circuit is $-V$, so we compare against $-(2|s\rangle\langle s| - I)$.

In [4]:
N = 2**n
s = np.ones(N) / np.sqrt(N)
V_book = 2 * np.outer(s, s) - np.eye(N)
V_circ = Operator(diffuser(n)).data
assert np.allclose(V_circ, -V_book), "diffuser wrong"
print("Test 2 passed: circuit = -V, i.e. V up to the global sign the book warns about")

Test 2 passed: circuit = -V, i.e. V up to the global sign the book warns about


## The full circuit and the choice of $r$

$r = \lfloor \tfrac{\pi}{4}\sqrt{N/\mu} \rfloor$, and stop there.

In [5]:
def grover_circuit(n, w, r, measure=True):
    qc = QuantumCircuit(n + 1, n)
    a = bq(n, n + 1)
    qc.x(a); qc.h(a)                              # ancilla |->
    qc.h([bq(j, n + 1) for j in range(n)])        # uniform superposition |s>
    for _ in range(r):
        qc.compose(oracle_known_target(n, w), inplace=True)
        qc.compose(diffuser(n), qubits=[bq(j, n + 1) for j in range(n)], inplace=True)
    if measure:
        # classical bit j holds book wire j, so the printed string reads in book order
        qc.measure([bq(j, n + 1) for j in range(n)], [n - 1 - j for j in range(n)])
    return qc


mu = 1
r_opt = int(np.floor(np.pi / 4 * np.sqrt(N / mu)))
print("n =", n, " N =", N, " r_opt =", r_opt)
print(grover_circuit(n, w, 1, measure=False).draw(fold=120))

n = 3  N = 8  r_opt = 2
     ┌───┐┌───┐┌───┐                                   
q_0: ┤ X ├┤ H ├┤ X ├───────────────────────────────────
     ├───┤└───┘└─┬─┘┌───┐┌───┐          ┌───┐┌───┐     
q_1: ┤ H ├───────■──┤ H ├┤ X ├───────■──┤ X ├┤ H ├─────
     ├───┤┌───┐  │  ├───┤├───┤┌───┐  │  ├───┤├───┤     
q_2: ┤ H ├┤ X ├──■──┤ X ├┤ H ├┤ X ├──■──┤ X ├┤ H ├─────
     ├───┤└───┘  │  ├───┤├───┤├───┤┌─┴─┐├───┤├───┤┌───┐
q_3: ┤ H ├───────■──┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├┤ X ├┤ H ├
     └───┘          └───┘└───┘└───┘└───┘└───┘└───┘└───┘
c: 3/══════════════════════════════════════════════════
                                                       


## Test 3: $\Pr(w)$ against $r$ for $n=3$, including the fall after the peak

The book's criterion: if the probability rises monotonically, the diffuser is wrong.

In [6]:
theta = np.arcsin(np.sqrt(mu / N))
rs = np.arange(0, 9)
p_sim, p_theory = [], []
for r in rs:
    sv = Statevector(grover_circuit(n, w, r, measure=False))
    # probability of the search register reading w, ancilla traced out
    probs = sv.probabilities([bq(j, n + 1) for j in range(n)][::-1])  # Qiskit order: LSB first
    # probs is indexed by the integer whose LSB is Qiskit qubit bq(n-1)... reorder to book integer
    p_sim.append(probs[w])
    p_theory.append(np.sin((2 * r + 1) * theta) ** 2)

for r, a, b in zip(rs, p_sim, p_theory):
    print(f"r={r}: simulated {a:.4f}   sin^2((2r+1)theta) {b:.4f}")
assert np.allclose(p_sim, p_theory, atol=1e-10)
print("Test 3 passed: matches sin^2((2r+1)theta); peak at r =", r_opt, "then it falls")

plt.plot(rs, p_sim, "o", label="circuit")
plt.plot(rs, p_theory, "-", label=r"$\sin^2((2r+1)\theta)$")
plt.xlabel("Grover iterations r"); plt.ylabel("Pr(w)"); plt.legend(); plt.show()

r=0: simulated 0.1250   sin^2((2r+1)theta) 0.1250
r=1: simulated 0.7812   sin^2((2r+1)theta) 0.7813
r=2: simulated 0.9453   sin^2((2r+1)theta) 0.9453
r=3: simulated 0.3301   sin^2((2r+1)theta) 0.3301
r=4: simulated 0.0122   sin^2((2r+1)theta) 0.0122
r=5: simulated 0.5480   sin^2((2r+1)theta) 0.5480
r=6: simulated 0.9998   sin^2((2r+1)theta) 0.9998
r=7: simulated 0.5770   sin^2((2r+1)theta) 0.5770
r=8: simulated 0.0195   sin^2((2r+1)theta) 0.0195
Test 3 passed: matches sin^2((2r+1)theta); peak at r = 2 then it falls


## Shot-based run on the Aer simulator

Only after the three matrix tests pass does the book say to run on hardware. Here the
"hardware" is the Aer simulator with shots; the `AerSimulator` line is where a real
backend would go.

In [7]:
sim = AerSimulator()
qc = grover_circuit(n, w, r_opt)
counts = sim.run(transpile(qc, sim), shots=2000).result().get_counts()
print("target w =", format(w, f"0{n}b"))
print(dict(sorted(counts.items(), key=lambda kv: -kv[1])))

target w = 101
{'101': 1886, '010': 26, '111': 24, '100': 20, '011': 14, '000': 14, '110': 9, '001': 7}


## Gate counts: what you are asking of the hardware

The book estimates about $4n$ Toffolis per iteration and $O(n\sqrt N)$ overall. The
transpiler's decomposition into a {u, cx} basis gives the two-qubit count that matters.

In [8]:
for nn in [3, 4, 5, 6]:
    rr = int(np.floor(np.pi / 4 * np.sqrt(2**nn)))
    qc = grover_circuit(nn, 1, rr, measure=False)
    t = transpile(qc, basis_gates=["u", "cx"], optimization_level=1)
    print(f"n={nn}: r={rr}, qubits={t.num_qubits}, cx={t.count_ops().get('cx', 0)}, depth={t.depth()}")

n=3: r=2, qubits=4, cx=40, depth=75
n=4: r=3, qubits=5, cx=150, depth=271


n=5: r=4, qubits=6, cx=408, depth=525


n=6: r=6, qubits=7, cx=984, depth=1247


The CX count grows by roughly $\sqrt{2}$ per added search qubit times the growth of the
multi-controlled gates, as the book says. Note Qiskit's `mcx` without explicit ancillas
uses an ancilla-free decomposition that is far longer than the clean-ancilla Toffoli
chain of Chapter 3; supplying `ancilla_qubits` and `mode='v-chain'` reproduces the
book's $2k-3$ Toffoli count.